# Bigram TF-IDF Exploration

Loads the pre-built bigram files from `data/interim/bigrams/` and computes corpus-level statistics for each bigram:

| Column | Formula |
|---|---|
| **tf** | $\sum_t f(b,t)$ — total count across all transcripts |
| **idf** | $\log\!\left(\dfrac{T}{df(b)}\right)$ |
| **tf-idf** | $tf \times idf$ |
| **tf-idf (norm)** | min-max normalised tf-idf over the displayed bigrams |

Choose `SEGMENT` to switch between `combined`, `pres`, and `answers`.

In [ ]:
import math
import csv
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_rows", 50)

PROJECT_ROOT = Path("..").resolve()
BIGRAM_DIR   = PROJECT_ROOT / "data" / "interim" / "bigrams"
PROCESSED    = PROJECT_ROOT / "data" / "processed"

# ── configuration ──────────────────────────────────────────────────────────
SEGMENT  = "combined"   # "combined" | "pres" | "answers"
TOP_N    = 50           # number of bigrams to display
# ───────────────────────────────────────────────────────────────────────────

SUFFIX_MAP = {
    "combined": ("_presentation.txt", "_answers.txt"),
    "pres":     ("_presentation.txt",),
    "answers":  ("_answers.txt",),
}
suffixes = SUFFIX_MAP[SEGMENT]
print(f"Segment: {SEGMENT}   Suffixes: {suffixes}")

## Load bigram files and compute corpus-level TF and DF

Scan every segment file for the chosen set. For each transcript aggregate bigram counts (TF) and track how many transcripts contain each bigram (DF).

In [ ]:
def _transcript_id(fname):
    for sfx in ("_presentation.txt", "_answers.txt", "_questions.txt"):
        if fname.endswith(sfx):
            return fname[: -len(sfx)]
    return None

# group segment files by transcript
seg_files: dict[str, dict[str, Path]] = defaultdict(dict)
for f in sorted(BIGRAM_DIR.iterdir()):
    tid = _transcript_id(f.name)
    if tid:
        for sfx in ("_presentation.txt", "_answers.txt", "_questions.txt"):
            if f.name.endswith(sfx):
                seg_files[tid][sfx] = f

transcript_ids = sorted(seg_files.keys())
T = len(transcript_ids)
print(f"Transcripts found: {T}")

# aggregate TF and DF
corpus_tf: Counter = Counter()   # total count across all docs
corpus_df: Counter = Counter()   # number of docs containing each bigram

for tid in transcript_ids:
    doc_counter: Counter = Counter()
    for sfx in suffixes:
        fpath = seg_files[tid].get(sfx)
        if fpath is None:
            continue
        text = fpath.read_text(encoding="utf-8").strip()
        if text:
            doc_counter.update(text.splitlines())
    corpus_tf.update(doc_counter)
    corpus_df.update(doc_counter.keys())

print(f"Unique bigrams in corpus: {len(corpus_tf):,}")

## Compute IDF, TF-IDF, and Normalized TF-IDF

$$IDF(b) = \log\!\left(\frac{T}{df(b)}\right), \quad TFIDF(b) = tf(b) \times IDF(b), \quad TFIDF_{norm}(b) = \frac{TFIDF(b) - \min}{\max - \min}$$

In [ ]:
# select top N bigrams by raw TF to build the display table
top_bigrams = corpus_tf.most_common(TOP_N)

rows = []
for bigram, tf_val in top_bigrams:
    df_val  = corpus_df[bigram]
    idf_val = math.log(T / df_val)
    tfidf   = tf_val * idf_val
    rows.append({
        "bigram":           bigram,
        "tf":               tf_val,
        "idf":              idf_val,
        "tf-idf":           tfidf,
    })

df_result = pd.DataFrame(rows)

# min-max normalise tf-idf over the displayed rows
tfidf_min = df_result["tf-idf"].min()
tfidf_max = df_result["tf-idf"].max()
df_result["tf-idf (norm)"] = (df_result["tf-idf"] - tfidf_min) / (tfidf_max - tfidf_min)

df_result